In [ ]:
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE documents_dummy (
    id SERIAL PRIMARY KEY,
    content TEXT,
    embedding vector(768)  -- adjust dimension to match your model
);


In [12]:
import psycopg2
import numpy as np
from langchain_ollama import OllamaEmbeddings
from psycopg2.extras import RealDictCursor

In [9]:
embeddings = OllamaEmbeddings(model="paraphrase-multilingual:278m")

In [13]:
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASS = "password"

def get_db_connection():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASS,
            cursor_factory=RealDictCursor
        )
        return conn
    except Exception as e:
        print(f"Database connection failure: {str(e)}")
        # raise HTTPException(
        #     status_code=500,
        #     detail=f"Database connection failure: {str(e)}"
        # )

In [22]:
conn = get_db_connection()
cur = conn.cursor()

In [29]:
## IF face error , in below insert, uncomment and run this
conn.rollback() 


In [26]:
docs = [
    "LangChain simplifies building applications with large language models.",
    "LangChain supports retrieval-augmented generation workflows.",
    "It provides tools for context-aware reasoning applications."
]

# Insert documents + embeddings
for doc in docs:
    emb = embeddings.embed_query(doc)  # returns a list of floats
    print(emb[0:5])
    emb_str = "[" + ",".join([str(x) for x in emb]) + "]"
    cur.execute("INSERT INTO documents_dummy (content, embedding) VALUES (%s, %s)", (doc, emb_str))

conn.commit()



[-0.035905384, -0.059922826, -0.0026280533, 0.04046721, -0.013981766]
[-0.016425224, 0.0004954776, -0.0041029244, 0.028771771, -0.013746034]
[-0.041093696, -0.058518812, -0.0019020394, 0.033653207, -0.0066921916]


In [ ]:
# Query embedding
query = "What is LangChain?"
query_emb = embeddings.embed_query(query)
query_emb_str = "[" + ",".join([str(x) for x in query_emb]) + "]"

# Similarity search using pgvector <=> operator (cosine distance)
cur.execute("""
    SELECT content, embedding <=> %s AS distance
    FROM documents_dummy
    ORDER BY distance ASC
    LIMIT 3;
""", (query_emb_str,))

results = cur.fetchall()

In [36]:
for r in results:
    print(f"Distance: {r['distance']}, 'Text': {r['content']}")

Distance: 0.366459894994955, 'Text': LangChain simplifies building applications with large language models.
Distance: 0.4420271101191029, 'Text': LangChain supports retrieval-augmented generation workflows.
Distance: 0.6876490135588883, 'Text': It provides tools for context-aware reasoning applications.


In [37]:
cur.close()
conn.close()